# MBA AttackerTrainer Validation

Валидация класса AttackerTrainer из src/training/train_attacker.py на примере LSTM-PatchTST-LSTM на PowerCons


## Шаг 1: Загрузка данных PowerCons


In [2]:
import sys
import os
# Перейти в корневую директорию проекта
os.chdir('..')
sys.path.append('.')

!ls

attack_run.py	   notebooks	     results
checkpoints	   notebooks_gen     run_full_experiments.sh
config		   outputs	     source
data		   __pycache__	     src
datasets	   pyproject.toml    tests
docker_scripts	   README_hid.md     train_attacker.py
experiment_params  README.md	     train_classifier.py
MBA_README.md	   requirements.txt  train_discriminator.py


In [3]:

from src.data import load_data, transform_data, MyDataset
from torch.utils.data import DataLoader
import torch

X_train, y_train, X_test, y_test = load_data('PowerCons')
X_train, X_test, y_train, y_test = transform_data(X_train, X_test, y_train, y_test, slice_data=False)
print(f"After transform_data - Train: {X_train.shape}, Test: {X_test.shape}")

# MyDataset автоматически добавляет размерность features в __getitem__ если len(X.shape) == 1
# Поэтому не нужно делать unsqueeze вручную

# Создать DataLoader с MyDataset (как в train_classifier.py)
train_loader = DataLoader(
    MyDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    MyDataset(X_test, y_test),
    batch_size=32,
    shuffle=False,
)


After transform_data - Train: torch.Size([180, 144]), Test: torch.Size([180, 144])


## Шаг 2: Загрузка обученного LSTM классификатора


In [4]:
from src.models import LSTM

# Найти улучшенную обученную модель (обучена через командную строку с переопределением параметров)
model_path = "checkpoints/PowerCons/LSTM/test/model_LSTM_400_PowerCons.pt"

# Загрузить улучшенную модель с лучшими параметрами
lstm_model = LSTM(hidden_dim=128, x_dim=1, activation_type="sigmoid", dropout=0.3, n_layers=2)
lstm_model.load_state_dict(torch.load(model_path, map_location='cpu'))
lstm_model.eval()
print("Improved LSTM model loaded (accuracy ~96.7%)")


CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


[KeOps] Warning : CUDA was detected, but driver API could not be initialized. Switching to CPU only.
Improved LSTM model loaded (accuracy ~96.7%)


## Шаг 3: Проверка работы LSTM на PowerCons


In [6]:
# Используем встроенный estimator как в оригинальном пайплайне
from src.estimation.estimators import ClassifierEstimator
import numpy as np

estimator = ClassifierEstimator(n_classes=2)

# Собираем все предсказания
y_all_pred_prob = torch.tensor([])
y_all_true = torch.tensor([])

with torch.no_grad():
    for x, y in test_loader:
        outputs = lstm_model(x)
        y_all_true = torch.cat((y_all_true, y.cpu().detach()), dim=0)
        y_all_pred_prob = torch.cat((y_all_pred_prob, outputs.cpu().detach()), dim=0)

# Подготавливаем данные для estimator
y_all_true = y_all_true.numpy().reshape(-1)
y_all_pred = y_all_pred_prob.round().numpy().reshape(-1)  # Бинарные предсказания
y_all_pred_prob = y_all_pred_prob.numpy().reshape(-1)     # Вероятности

# Вычисляем метрики через estimator
metrics = estimator.estimate(y_all_true, y_all_pred, y_all_pred_prob)
metric_names = estimator.get_metrics_names()

# Выводим результаты
print("LSTM Metrics:")
for name, value in zip(metric_names, metrics):
    print(f"  {name}: {value:.4f}")

accuracy = metrics[0]  # accuracy - первый элемент
print(f"\nLSTM Accuracy: {accuracy:.4f}")

# Проверка
assert accuracy > 0.8, f"LSTM accuracy {accuracy:.4f} должна быть > 0.8!"
print("✓ LSTM работает корректно")


LSTM Metrics:
  accuracy: 0.9667
  precision: 0.9884
  recall: 0.9444
  f1: 0.9659
  balance_true: 0.5000
  balance_pred: 0.4778
  certainty: 0.9647

LSTM Accuracy: 0.9667
✓ LSTM работает корректно


## Шаг 4: Инициализация AttackPatchTST


In [ ]:
from src.models import AttackPatchTST

# Параметры из конфига config/attacker_model/AttackPatchTST.yaml
attacker_model = AttackPatchTST(
    hidden_dim=128,
    x_dim=1,
    activation_type="tanh",
    patch_kwargs={
        'seq_len': 144,
        'n_layers': 3,
        'n_heads': 8,
        'd_model': 512,
        'd_ff': 2048,
        'dropout': 0.4,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': True,
        'revin': True,
        'res_attention': True
    }
)
print("AttackPatchTST initialized")


## Шаг 5: Проверка forward pass AttackPatchTST


In [ ]:
# Тест на одном батче
test_batch = torch.tensor(X_train[:4], dtype=torch.float32)  # (4, 144, 1)
output = attacker_model(test_batch)
print(f"Input shape: {test_batch.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == test_batch.shape, "Output shape mismatch!"
print("✓ Forward pass OK")


## Шаг 6: Инициализация AttackerTrainer


In [ ]:
from src.training.train_attacker import AttackerTrainer
import torch.nn as nn

# Параметры из config/train_attacker_config.yaml
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(attacker_model.parameters(), lr=1e-4)
# В source версии scheduler НЕ используется!
scheduler = None

# Создать AttackerTrainer
trainer = AttackerTrainer(
    attacker_model=attacker_model,
    victim_model=lstm_model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    n_epochs=10,
    early_stop_patience=4,
    logger=None,
    print_every=1,
    device='cpu',
    eps=0.5,
    alpha_l2=1e-3,
    is_clamped=False,
    is_debugged=False
)
print("✓ AttackerTrainer initialized")


## Шаг 7: Проверка train_epoch


In [ ]:
# Проверить один epoch обучения
print("\n=== Testing train_epoch ===")
train_metrics = trainer.train_epoch(train_loader)

print(f"Train metrics:")
print(f"  victim_loss: {train_metrics['victim_loss']:.4f}")
print(f"  accuracy: {train_metrics['accuracy']:.4f}")
print(f"  regularization: {train_metrics['regularization']:.6f}")
print(f"  attacker_loss: {train_metrics['attacker_loss']:.4f}")

# Проверки
assert train_metrics['victim_loss'] > 0, "victim_loss должен быть > 0!"
assert 0 <= train_metrics['accuracy'] <= 1, "accuracy должен быть в [0, 1]!"
assert train_metrics['regularization'] >= 0, "regularization должен быть >= 0!"
print("\n✓ train_epoch работает корректно")


## Шаг 8: Проверка validate_epoch


In [ ]:
print("\n=== Testing validate_epoch ===")
val_metrics = trainer.validate_epoch(test_loader)

print(f"Validation metrics:")
print(f"  victim_loss: {val_metrics['victim_loss']:.4f}")
print(f"  accuracy: {val_metrics['accuracy']:.4f}")

assert val_metrics['victim_loss'] > 0, "victim_loss должен быть > 0!"
assert 0 <= val_metrics['accuracy'] <= 1, "accuracy должен быть в [0, 1]!"
print("\n✓ validate_epoch работает корректно")


## Шаг 9: Полное обучение через AttackerTrainer.train()


In [ ]:
print("\n=== Testing full training ===")

# Переинициализировать модель для чистого теста
attacker_model = AttackPatchTST(
    hidden_dim=128,
    x_dim=1,
    activation_type="tanh",
    patch_kwargs={
        'seq_len': 144,
        'n_layers': 3,
        'n_heads': 8,
        'd_model': 512,
        'd_ff': 2048,
        'dropout': 0.4,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': True,
        'revin': True,
        'res_attention': True
    }
)

optimizer = torch.optim.Adam(attacker_model.parameters(), lr=1e-4)
# В source версии scheduler НЕ используется!
scheduler = None

trainer = AttackerTrainer(
    attacker_model=attacker_model,
    victim_model=lstm_model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    n_epochs=10,
    early_stop_patience=4,
    logger=None,
    print_every=1,
    device='cpu',
    eps=0.5,
    alpha_l2=1e-3,
    is_clamped=False,
    is_debugged=False
)

# Запустить обучение
results = trainer.train(train_loader, test_loader)

print("\n=== Training Results ===")
print(f"Best validation loss: {results['best_val_loss']:.4f}")
print(f"Number of train epochs: {len(results['history']['train'])}")
print(f"Number of val epochs: {len(results['history']['val'])}")


## Шаг 10: Анализ динамики обучения


In [ ]:
import matplotlib.pyplot as plt

# Извлечь метрики по эпохам
train_vloss = [m['victim_loss'] for m in results['history']['train']]
val_vloss = [m['victim_loss'] for m in results['history']['val']]
train_acc = [m['accuracy'] for m in results['history']['train']]
val_acc = [m['accuracy'] for m in results['history']['val']]

# Построить графики
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_vloss, label='Train victim loss')
ax1.plot(val_vloss, label='Val victim loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Victim Loss')
ax1.set_title('Victim Loss Dynamics')
ax1.legend()
ax1.grid(True)

ax2.plot(train_acc, label='Train victim acc')
ax2.plot(val_acc, label='Val victim acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Victim Accuracy')
ax2.set_title('Victim Accuracy Dynamics')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Проверки
print("\n=== Validation Checks ===")
print(f"Initial victim loss: {val_vloss[0]:.4f}")
print(f"Final victim loss: {val_vloss[-1]:.4f}")
print(f"Loss increased: {val_vloss[-1] > val_vloss[0]}")

print(f"\nInitial victim accuracy: {val_acc[0]:.4f}")
print(f"Final victim accuracy: {val_acc[-1]:.4f}")
print(f"Accuracy decreased: {val_acc[-1] < val_acc[0]}")

# Критерии успеха
assert val_vloss[0] > 0, "Initial victim loss должен быть > 0!"
assert val_vloss[-1] > val_vloss[0], "Victim loss должен расти (атака обучается)!"
assert val_acc[-1] < val_acc[0], "Victim accuracy должен падать (атака работает)!"
print("\n✓ Все проверки пройдены!")


## Шаг 11: Проверка возмущений


In [ ]:
print("\n=== Perturbation Analysis ===")
attacker_model.eval()
with torch.no_grad():
    test_batch = torch.tensor(X_test[:10], dtype=torch.float32)
    delta = trainer.eps * torch.tanh(attacker_model(test_batch))
    
    # Статистика возмущений
    delta_norm = torch.norm(delta.view(delta.size(0), -1), p=2, dim=1)
    print(f"Average L2 norm: {delta_norm.mean():.4f}")
    print(f"Max L2 norm: {delta_norm.max():.4f}")
    print(f"Min L2 norm: {delta_norm.min():.4f}")
    print(f"Max perturbation value: {delta.abs().max():.4f}")
    print(f"Min perturbation value: {delta.abs().min():.6f}")
    
    assert delta_norm.mean() > 0, "Возмущения не должны быть нулевыми!"
    print("\n✓ Возмущения генерируются корректно")


## Шаг 12: Диагностика проблемы loss=0


In [ ]:
print("\n=== Criterion Diagnostics ===")

# Проверить формат выхода LSTM
test_input = torch.tensor(X_test[:4], dtype=torch.float32)
lstm_output = lstm_model(test_input)
print(f"LSTM output shape: {lstm_output.shape}")
print(f"LSTM output range: [{lstm_output.min():.4f}, {lstm_output.max():.4f}]")
print(f"LSTM output dtype: {lstm_output.dtype}")

# Проверить criterion
print(f"\nCriterion: {criterion}")
print(f"Expected format for CrossEntropyLoss: (batch, num_classes) или (batch,) для бинарной")
print(f"Expected format for BCELoss: (batch,) с sigmoid выходом в [0, 1]")

# Проверить loss вычисление
test_labels = torch.tensor(y_test[:4], dtype=torch.long)
try:
    loss_ce = nn.CrossEntropyLoss()(lstm_output, test_labels)
    print(f"\nCrossEntropyLoss: {loss_ce.item():.4f}")
except Exception as e:
    print(f"\nCrossEntropyLoss error: {e}")

try:
    loss_bce = nn.BCELoss()(lstm_output.squeeze(), test_labels.float())
    print(f"BCELoss: {loss_bce.item():.4f}")
except Exception as e:
    print(f"BCELoss error: {e}")


## Шаг 13: Проверка градиентов


In [ ]:
print("\n=== Gradient Check ===")
attacker_model.train()
test_batch = torch.tensor(X_train[:4], dtype=torch.float32)
test_labels = torch.tensor(y_train[:4], dtype=torch.long)

delta = trainer.eps * torch.tanh(attacker_model(test_batch))
x_adv = test_batch + delta

logits = lstm_model(x_adv)
victim_loss = criterion(logits, test_labels)
reg = trainer.alpha_l2 * (delta**2).mean()
loss = -(victim_loss - reg)

loss.backward()

# Проверить градиенты
grad_norms = []
grad_info = []
for name, param in attacker_model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_norms.append(grad_norm)
        grad_info.append((name, grad_norm))

print(f"Gradient statistics:")
print(f"  Min: {min(grad_norms):.6f}")
print(f"  Max: {max(grad_norms):.6f}")
print(f"  Mean: {sum(grad_norms)/len(grad_norms):.6f}")

print(f"\nTop 5 gradients by norm:")
for name, norm in sorted(grad_info, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {name}: {norm:.6f}")

assert min(grad_norms) > 0, "Все градиенты должны быть > 0!"
print("\n✓ Градиенты вычисляются корректно")


## Результаты валидации

### Критерии успеха:
1. ✅ LSTM классификатор загружается и работает (accuracy > 0.8)
2. ✅ AttackPatchTST инициализируется и делает forward pass без ошибок
3. ✅ AttackerTrainer инициализируется без ошибок
4. ✅ train_epoch выполняется и возвращает корректные метрики (victim_loss > 0)
5. ✅ validate_epoch выполняется и возвращает корректные метрики
6. ✅ Полное обучение через train():
   - victim_loss растет от эпохи к эпохе
   - victim_accuracy падает от эпохи к эпохе
   - Возмущения не нулевые
   - Градиенты вычисляются корректно

### Если тесты не проходят:
- Документировать в ячейках выше:
  - Какой именно тест не прошел
  - Фактические vs ожидаемые значения
  - Размерности тензоров на каждом шаге
  - Значения loss и метрик
  - Traceback ошибок
